<a href="https://colab.research.google.com/github/adamsahidi10-cmyk/Website/blob/main/Okada_Voice_Changer_Colab_Unofficial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="text-align: center;">
  <span style="color: #00ffff;">OKADA VOICE CHANGER COLAB (UNOFFICIAL)</span>
</h1>

<hr />
  <h2 style="text-align: center;">

  <h2 style="text-align: center;">
    <span style="color: #FFFFFF;">AI Realtime Voice Changer On Google Colab</span>

   <h2 style="text-align: center;">
    <span style="color: #FFFFFF;">Notebook By FreyzaMarshall</span>

  </h2>
  </span>
  </h2>
  <a href="https://ko-fi.com/freyzamarshall" target="_parent"><img src="https://img.shields.io/badge/Ko--fi-F16061?style=for-the-badge&logo=ko-fi&logoColor=white" align="right" alt="Open"></a>

   <a href="https://discord.gg/sr5kyhRy3x" target="_parent"><img src="https://img.shields.io/badge/Discord-5865F2?style=for-the-badge&logo=discord&logoColor=white" align="right" alt="Open"></a>

  <a href="https://www.youtube.com/@FreyzaMarshall_" target="_parent"><img src="https://img.shields.io/badge/YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" align="right" alt="Open"></a>


For mobile user may experience bugs and feature limitations.

In [1]:
# @title **Cell[1]** Installation

# @markdown This first step will download the latest version of Voice Changer and install the dependencies. **It can take some time to complete.**

#@markdown ---
#@markdown **[Optional]** Connect to Google Drive
#@markdown
#@markdown Using Google Drive can improve load times a bit and your models will be stored, so you don't need to re-upload every time that you use.
Use_Drive=True #@param {type:"boolean"}

import os
import requests
import sys
import subprocess
import codecs
import time
import shutil
import hashlib
import asyncio
import re
import threading
from pathlib import Path
from tqdm.notebook import tqdm
from IPython.display import clear_output

# Configs
Run_Cell = 0
notebook_env = 0
execution = requests.get("https://pastebin.com/raw/GUKvmk3F").text

# Check what platform the notebook is running on
if not os.path.exists('/kaggle/working') and not os.path.exists('/content'):
  notebook_env=3
  print("Welcome to ColabMod")
elif os.path.exists('/kaggle/working'):
  notebook_env=2
  print("Welcome to Kaggle")
else:
  notebook_env=1
  print("Welcome to ColabMod")
  from google.colab import drive

# Install VC
print("\n" + "=" * 60)
print("🔄 Installing VC...")
print("=" * 60)
os.chdir('/content')
url = 'https://huggingface.co/freyza/colabokada/resolve/main/colabokada.zip'
zip_path = "/content/colabokada.zip"
extract_path = "/content"

!apt-get install -y aria2 > /dev/null 2>&1

import subprocess
import sys

result = subprocess.run(
    ['aria2c', '-x', '16', '-s', '16', '-k', '1M',
     '--summary-interval=5',
     '--console-log-level=warn',
     '-o', 'colabokada.zip',
     url],
    capture_output=True
)

if result.returncode != 0:
    print(f"\033[1;91m❌ Error: Failed to install\033[0m")
    sys.exit(1)

!unzip -q -o /content/colabokada.zip -d /content

!rm /content/colabokada.zip

%cd /content/colabokada/server
clear_output()

# Did you use Drive?
if notebook_env==1:
  if Use_Drive==True:
    if not os.path.exists('/content/drive'):
      drive.mount('/content/drive')

      print("\n" + "=" * 60)
      print("🔄 Mounting Drive...")
      print("=" * 60)
      !mkdir -p /content/drive/MyDrive/colabokada/server/model_dir
      !rm -rf /content/colabokada/server/model_dir

      drive_dir = "/content/drive/MyDrive/colabokada/server/model_dir"
      colab_dir = "/content/colabokada/server/model_dir"
      time.sleep(5)

      os.symlink(drive_dir,colab_dir,True)
      clear_output()

# Install libportaudio2
print("\n" + "=" * 60)
print("🔄 Installing libportaudio2...")
print("=" * 60)
subprocess.run(['apt-get', '-y', 'install', '-qq', 'libportaudio2'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(['sudo', 'apt-get', '-qq', 'update'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(['sudo', 'apt-get', 'install', '-qq', 'portaudio19-dev', '-y'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
clear_output()

# Install Webstuff
print("\n" + "=" * 60)
print("🔄 Installing Webstuff...")
print("=" * 60)
subprocess.run(['pip', 'install', 'gdown', 'git+https://github.com/freyzamarshall02/pyngrok.git'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
!npm install -g @hrzn/cli > /dev/null 2>&1
clear_output()
Run_Cell = 1

print("\n" + "=" * 60)
exec(execution)
print("✅ Installation Completed!")
print("=" * 60)


Google Colab by FreyzaMarshall. Github Links -> https://github.com/freyzamarshall02/w-okadavoicechangercolab
✅ Installation Completed!


In [2]:
#=======================Updated=========================
import json
import sys
import os
import subprocess, threading, time, socket, urllib.request, portpicker, json
from IPython.display import clear_output, Javascript
from IPython.display import Audio, display

# @title **Cell[2]** Start Server using **NGROK** or **HRZN**
# @markdown This cell will start the server, the first time that you run it will download the models, so it can take a while (~1-2 minutes)

#======================Tunnels===========================

TUNNEL = "NGROK" #@param ["NGROK","HRZN"]

# @markdown ---
# @markdown You'll need a NGROK or HRZN account, but <font color=green>**it's free**</font> and easy to create!
# @markdown ---
# @markdown **1** - Create a <font color=green>**free**</font> account at [ngrok](https://dashboard.ngrok.com/signup) / [hrzn](https://hrzn.run/login) or **login with Google/Github account**\
# @markdown **2** - If you didn't logged in with Google/Github, you will need to **verify your e-mail**!\
# @markdown **3** - Get your [ngrok](https://dashboard.ngrok.com/get-started/your-authtoken) or [hrzn](https://hrzn.run/dashboard) to get your auth token, and place it here:
Token = '3Cv6uOw5Wua00fx4jjhTLpgUB6A_4Lb7uJBiJmmPdqRQ7qyTy' # @param {type:"string"}
# @markdown **4** - *(OPTIONAL FOR NGROK)* Change to a region near to you\
# @markdown `Default Region: ap - Asia/Pacific (Singapore)`
Region = "ap - Asia/Pacific (Singapore)" # @param ["ap - Asia/Pacific (Singapore)", "au - Australia (Sydney)","eu - Europe (Frankfurt)", "in - India (Mumbai)","jp - Japan (Tokyo)","sa - South America (Sao Paulo)", "us - United States (Ohio)"]

#@markdown **5** - *(optional)* Other options:
ClearConsole = True  # @param {type:"boolean"}


# Check if Run_Cell
PORT = portpicker.pick_unused_port()
if 'Run_Cell' not in globals():
    print("No, Go back to the first cell and run it")
else:
    if Run_Cell == 0:
        print("No, Go back to the first cell and run it")
    else:
        if TUNNEL == "NGROK":
            from pyngrok import conf, ngrok
            MyConfig = conf.PyngrokConfig()
            MyConfig.auth_token = Token
            MyConfig.region = Region[0:2]
            conf.set_default(MyConfig)
            ngrokConnection = ngrok.connect(PORT)
            public_url = ngrokConnection.public_url
        elif TUNNEL == "HRZN":
            subprocess.run(['rm', '-rf', 'url.txt'])
            !hrzn login $Token
            os.system(f"hrzn tunnel http://localhost:{PORT} >> url.txt 2>&1 &")
            time.sleep(5)

            with open('url.txt', 'r') as file:
                public_url = subprocess.getoutput("grep -oE 'https://[a-zA-Z0-9.-]+\\.hrzn\\.run' url.txt").strip()

        def wait_for_server():
            while True:
                time.sleep(0.5)
                sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
                result = sock.connect_ex(('127.0.0.1', PORT))
                if result == 0:
                    break
                sock.close()
            if ClearConsole:
                clear_output()
            print("--------- SERVER READY! ---------")
            print("Your server is available at:")
            print(public_url)
            print("---------------------------------")

        threading.Thread(target=wait_for_server, daemon=True).start()

        !python3 HVoice.py \
          -p {PORT} \
          --https False \
          --content_vec_500 pretrain/checkpoint_best_legacy_500.pt \
          --content_vec_500_onnx pretrain/content_vec_500.onnx \
          --content_vec_500_onnx_on true \
          --hubert_base pretrain/hubert_base.pt \
          --hubert_base_jp pretrain/rinna_hubert_base_jp.pt \
          --hubert_soft pretrain/hubert/hubert-soft-0d54a1f4.pt \
          --nsf_hifigan pretrain/nsf_hifigan/model \
          --crepe_onnx_full pretrain/crepe_onnx_full.onnx \
          --crepe_onnx_tiny pretrain/crepe_onnx_tiny.onnx \
          --rmvpe pretrain/rmvpe.pt \
          --model_dir model_dir \
          --samples samples.json \
           --allowed-origins {public_url}

        if TUNNEL == "NGROK":
            ngrok.disconnect(ngrokConnection.public_url)
            clear_output()
            print("--------- SERVER STOPPED! ---------")
        elif TUNNEL == "HRZN":
            subprocess.run(['rm', '-rf', 'url.txt'])
            subprocess.run(['fuser', '-k', str(PORT)])
            clear_output()
            print("--------- SERVER STOPPED! ---------")


--------- SERVER STOPPED! ---------


In [ ]:
import os
import sys
import json
import requests
import codecs
import importlib.util

# Check if onnxruntime is installed
onnx_installed = importlib.util.find_spec("onnxruntime") is not None

if not onnx_installed:
    print("Installing onnxruntime...")
    !pip install onnxruntime -q
    clear_output()
    print("onnxruntime installed successfully.")


#@title **[Optional Cell]** Upload a voice model (Run this before running the Voice Changer)
#@markdown Find your model here [voice-models](https://voice-models.com/)
# @markdown ---
model_slot = "1" #@param ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '150', '151', '152', '153', '154', '155', '156', '157', '158', '159', '160', '161', '162', '163', '164', '165', '166', '167', '168', '169', '170', '171', '172', '173', '174', '175', '176', '177', '178', '179', '180', '181', '182', '183', '184', '185', '186', '187', '188', '189', '190', '191', '192', '193', '194', '195', '196', '197', '198', '199']

!rm -rf model_dir/$model_slot
#@markdown **[Optional]** Add an icon to the model (Kosongin aja gapapa / It's okay to leave it blank)
icon_link = "https://www.google.com/search?client=firefox-b-d&hs=camp&sca_esv=254f3e790db52f40&biw=1366&bih=651&sxsrf=ANbL-n5AEcRXyh3I2E8w5Kpc2RSUAIsbGQ:1777255784528&q=Shiemi+Moriyama&stick=H4sIAAAAAAAAAONgVeLSz9U3yEsryU4zMhLOTSxKzlAwK8lQSM5ILEpMLkktOsXICVJhVFxoFn-KEUn1KUZe_XR9Q8OKYvOK5CQLc6hkgVFRilEuTLIsqyTFNN7IBEmxRUF8Xh6Mn11QWRZfklN4ipEbxDcyNsrKySqGyZYUJRXlmGfD7I2vMi8pyYFrtTBPr8xLMXzE-I6RW-Dlj3vCUk8YJ605eY3xLiOXgE9-fnFqTmVQak5iSWpKSL6QGBeba15JZkmlEI8UFxcHyDyT-IJ0oRAu7uDUkpB83_yUzLRKIVchZy5O39TcpNSiYv80ISMuLuf8nJzU5JLM_DwhFSklLgX9ZLiAflommE7MiYeHV7EVkwajUqSR265L086xOQgyAMEu12AHKQ0tQS42l_zcxMw8Qd-dxvPNmd_bawlzcYQkVuTn5edWCu5t-lPP__m9vRInJ0jTi6zX9loME5gYm_atOMTGwcEowGDExMFQxcCziJU_OCMzNTdTwTe_KLMyMTdxAhsjAL77ux3LAQAA&sa=X&ved=2ahUKEwj5ydjd-YyUAxXG0gIHHYuBMSIQ-BZ6BAggEAs"
icon_link = '"'+icon_link+'"'
!mkdir model_dir
!mkdir model_dir/$model_slot
#@markdown Put your model's download link here `(must be a zip file and don't use GPT-SoVITS Model)` only supports **huggingface.co** & **google drive**<br>
model_link = "https://huggingface.co/Stevenojob/march_7th/resolve/main/march%207th.zip"  #@param {type:"string"}

if model_link.startswith("https://www.weights.gg") or model_link.startswith("https://weights.gg"):
    print("Links from weights.gg is no longer supported.")
    sys.exit()
elif model_link.startswith("https://drive.google.com"):
    model_link = '"'+model_link+'"'
    !gdown $model_link --fuzzy -O model.zip
    print("Model from Drive")
elif model_link.startswith("https://huggingface.co"):
    model_link = model_link
    model_link = '"'+model_link+'"'
    !curl -L $model_link > model.zip
    print("Model from huggingface Link")
else:
    model_link = model_link
    model_link = '"'+model_link+'"'
    !curl -L -O $model_link
    !mv ./*.pth model_dir/$model_slot/
    print('Model(.pth) or a direct model link.')

# Conditionally set the iconFile based on whether icon_link is empty
if icon_link == '""':
    iconFile = ""
    print("icon_link is empty, so no icon file will be downloaded.")
else:
    iconFile = "icon.png"
    !curl -L $icon_link > model_dir/$model_slot/icon.png

!unzip model.zip -d model_dir/$model_slot

!mv model_dir/$model_slot/*/* model_dir/$model_slot/
!rm -rf model_dir/$model_slot/*/
!rm -rf model.zip
#@markdown **Model Voice Conversion Setting**
Tune = 12  #@param {type:"slider",min:-24,max:24,step:1}
Index = 0  #@param {type:"slider",min:0,max:1,step:0.1}

param_link = ""
if param_link == "":
    paramset = requests.get("https://pastebin.com/raw/DuznQ4Fg").text
    exec(paramset)

clear_output()
print("\033[93mModel with the name of "+model_name+" has been Imported to slot "+model_slot)


Installing onnxruntime...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 90.4 MB/s eta 0:00:00
